<a href="https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**The Time-Aware Walk-Forward Validation**

Up until now, we trained and tested on the *same calendar window* (just with different clients held out). But in reality, we train on the past to predict the future. 

**The Test:** Train the exact same Random Forest on an earlier time window (e.g., December to January), and test it on our target March 2026 data.

**The Results (The Collapse):**
- The mean Precision@50 fell from **0.672** to **0.260**.
- That is effectively indistinguishable from the whole-frame base rate (0.25) and below the mean fold base rate (0.29).
- In 4 out of 5 folds, the model performed worse than its own fold's base rate.

**Why did this happen?**
Search traffic is inherently volatile and mean-reverting. Feature correlations literally flipped signs between the training window and the testing window. The model was aggressively memorizing local volatility spikes rather than durable decay patterns.

In [1]:
import os, getpass, duckdb, hashlib, json
import pandas as pd, numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF READ token (hf_...): ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Anchor output path to repo root regardless of notebook CWD
REPO_ROOT = os.getcwd()
while not os.path.exists(os.path.join(REPO_ROOT, 'AGENTS.md')) and os.path.dirname(REPO_ROOT) != REPO_ROOT:
    REPO_ROOT = os.path.dirname(REPO_ROOT)
OUT_DIR = os.path.join(REPO_ROOT, 'work', 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

REL = 'hf://datasets/FlyRank/internship-warehouse'
FM  = lambda m: f"read_parquet('{REL}/fact_content_daily_performance/month=2026-{m}/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Build feature + label table (same for all notebooks)
df = con.sql(f"""
    WITH per_content AS (
        SELECT
            f.content_hash_id, f.client_hash_id,
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_clicks    ELSE 0 END) AS clk_prev30,
            AVG(CASE  WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_avg_position END)   AS pos_prev30,
            COUNT(DISTINCT CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01'
                                 AND f.gsc_impressions > 0 THEN f.report_date END)                                                 AS days_with_imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2026-03-01' AND f.report_date < DATE '2026-04-01' THEN f.gsc_impressions ELSE 0 END) AS imp_last30
        FROM (SELECT * FROM {FM('01')} UNION ALL SELECT * FROM {FM('02')} UNION ALL SELECT * FROM {FM('03')}) f
        WHERE f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-04-01'
          AND f.gsc_data_available IS TRUE
        GROUP BY f.content_hash_id, f.client_hash_id
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM per_content
""").df()

meta = con.sql(f"""
    SELECT content_hash_id,
           DATEDIFF('day', content_created_date, DATE '2026-03-01') AS content_age_days,
           word_count, content_type
    FROM {DIM_CONTENT}
""").df()

df = df.merge(meta, on='content_hash_id', how='left')
df['is_declining'] = (df['imp_last30'] < 0.8 * df['imp_prev30']).astype(int)

# Log-scale heavy-tailed features; fill missing with 0 (documented)
df['pos_prev30']        = df['pos_prev30'].fillna(0)
df['content_age_days']  = df['content_age_days'].fillna(0)
df['log_imp_prev30']    = np.log1p(df['imp_prev30'])
df['log_clk_prev30']    = np.log1p(df['clk_prev30'])

FEATURES = ['log_imp_prev30', 'log_clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']

# Deterministic 5-way client hash fold (same contract across all notebooks)
def client_fold(cid, n=5):
    return int(hashlib.sha256(cid.encode()).hexdigest(), 16) % n

df['fold'] = df['client_hash_id'].map(client_fold)
df['_tie'] = df['content_hash_id'].map(lambda c: int(hashlib.sha256(c.encode()).hexdigest(), 16))

def precision_at_k(sorted_labels, k):
    head = sorted_labels.head(min(k, len(sorted_labels)))
    return float(head.mean()) if len(head) else float('nan')

print(f"Frame: {len(df):,} pages | {df['client_hash_id'].nunique()} clients")
print(f"Decline rate: {df['is_declining'].mean():.3f}")
print(f"Features (all pre-March-1): {FEATURES}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Frame: 81,521 pages | 37 clients
Decline rate: 0.249
Features (all pre-March-1): ['log_imp_prev30', 'log_clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']


**Leakage Audit:**
- **Denominator check:** `log_imp_prev30` is mathematically part of the label calculation. Dropping it slightly alters performance but doesn't cause a systemic collapse. Leakage exists but isn't load-bearing.
- **Window check:** Assertions in the code prove feature windows strictly close before label windows.

**Final Decision:**
The ML model does not survive a forward leap in time. Deploying it would yield results worse than random guessing. Therefore, **we will not ship the model**. We will ship the transparent heuristic rule (which proved robust across time) paired with a governed human-review playbook.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# Time-aware walk-forward validation

FM_TRAIN = lambda m: f"SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={m}/**/*.parquet')"

df_train_raw = con.sql(f"""
    WITH per_content AS (
        SELECT
            f.content_hash_id, f.client_hash_id,
            SUM(CASE WHEN f.report_date >= DATE '2025-11-30' AND f.report_date < DATE '2026-01-30' THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2025-11-30' AND f.report_date < DATE '2026-01-30' THEN f.gsc_clicks    ELSE 0 END) AS clk_prev30,
            AVG(CASE  WHEN f.report_date >= DATE '2025-11-30' AND f.report_date < DATE '2026-01-30' THEN f.gsc_avg_position END)   AS pos_prev30,
            COUNT(DISTINCT CASE WHEN f.report_date >= DATE '2025-11-30' AND f.report_date < DATE '2026-01-30'
                                 AND f.gsc_impressions > 0 THEN f.report_date END)                                                  AS days_with_imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_impressions ELSE 0 END) AS imp_last30
        FROM ({FM_TRAIN('2025-12')} UNION ALL {FM_TRAIN('2026-01')} UNION ALL {FM_TRAIN('2026-02')}) f
        WHERE f.report_date >= DATE '2025-11-30' AND f.report_date < DATE '2026-03-01'
          AND f.gsc_data_available IS TRUE
        GROUP BY f.content_hash_id, f.client_hash_id
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM per_content
""").df()

meta2 = con.sql(f"""
    SELECT content_hash_id,
           DATEDIFF('day', content_created_date, DATE '2026-01-30') AS content_age_days
    FROM {DIM_CONTENT}
""").df()
df_train_raw = df_train_raw.merge(meta2, on='content_hash_id', how='left')
df_train_raw['is_declining'] = (df_train_raw['imp_last30'] < 0.8 * df_train_raw['imp_prev30']).astype(int)

for col in ['pos_prev30', 'content_age_days']:
    df_train_raw[col] = df_train_raw[col].fillna(0)
df_train_raw['log_imp_prev30'] = np.log1p(df_train_raw['imp_prev30'])
df_train_raw['log_clk_prev30'] = np.log1p(df_train_raw['clk_prev30'])
df_train_raw['fold'] = df_train_raw['client_hash_id'].map(client_fold)
df_train_raw['_tie'] = df_train_raw['content_hash_id'].map(
    lambda c: int(hashlib.sha256(c.encode()).hexdigest(), 16))

from sklearn.ensemble import RandomForestClassifier
SEED = 42

# Walk-forward: train on Dec-Jan frame, test on March frame (df, already built)
ta_rows = []
for f in sorted(df['fold'].unique()):
    tr_raw = df_train_raw[df_train_raw['fold'] != f]
    te_mar = df[df['fold'] == f].copy()

    X_tr = tr_raw[FEATURES].fillna(0).to_numpy()
    y_tr = tr_raw['is_declining'].to_numpy()
    X_te = te_mar[FEATURES].fillna(0).to_numpy()

    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
    rf.fit(X_tr, y_tr)

    proba  = rf.predict_proba(X_te)[:, 1]
    te_mod = te_mar.assign(ta_score=proba).sort_values(['ta_score', '_tie'], ascending=[False, True]).reset_index(drop=True)

    rec = {'fold': int(f), 'n_test': int(len(te_mar)),
           'train_n': int(len(tr_raw)),
           'base_rate': round(float(te_mar['is_declining'].mean()), 4)}
    for k in (20, 50, 100):
        rec[f'time_aware_precision@{k}'] = round(precision_at_k(te_mod['is_declining'], k), 4)
    ta_rows.append(rec)

ta_df = pd.DataFrame(ta_rows)
print('=== Time-aware walk-forward audit (train Dec-Jan, test March) ===')
hdr = ['base_rate', 'time_aware_precision@20', 'time_aware_precision@50', 'time_aware_precision@100']
print(ta_df[['fold'] + hdr].to_string(index=False))
print()
for col in hdr:
    print(f'{col:35s} mean {ta_df[col].mean():.4f}')
print()
print('Comparison (same-window forest mean P@50 from W5 receipt):')
try:
    mv = json.load(open(os.path.join(OUT_DIR, 'model_vs_baseline_folds.json')))
    print(f"  same-window forest: {mv['mean_precision@50']['forest']:.4f}")
except FileNotFoundError:
    print('  (run w05 first to generate the receipt)')
print(f"  time-aware forest : {ta_df['time_aware_precision@50'].mean():.4f}")
print()
print('FINDING: the same-window win does NOT survive a forward leap in time.')
print('The model overfit to local volatility, not durable decay patterns.')
print('=> Decision: ship the transparent rule + human-review playbook, not the model.')

receipt = {
    'train_window': '2025-11-30 to 2026-01-30 (Dec-Jan features, Jan-Feb label)',
    'test_window': 'March 2026 (same frame as w04/w05)',
    'folds': ta_rows,
    'mean_precision@50_time_aware': round(float(ta_df['time_aware_precision@50'].mean()), 4),
}
with open(os.path.join(OUT_DIR, 'w06_validation_audit_receipt.json'), 'w') as fh:
    json.dump(receipt, fh, indent=2)
print('Receipt written.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Time-aware walk-forward audit (train Dec-Jan, test March) ===
 fold  base_rate  time_aware_precision@20  time_aware_precision@50  time_aware_precision@100
    0     0.2153                     0.15                     0.20                      0.14
    1     0.1284                     0.00                     0.04                      0.10
    2     0.6649                     0.45                     0.56                      0.69
    3     0.2743                     0.35                     0.36                      0.38
    4     0.1859                     0.10                     0.14                      0.16

base_rate                           mean 0.2938
time_aware_precision@20             mean 0.2100
time_aware_precision@50             mean 0.2600
time_aware_precision@100            mean 0.2940

Comparison (same-window forest mean P@50 from W5 receipt):
  same-window forest: 0.6720
  time-aware forest : 0.2600

FINDING: the same-window win does NOT survive a forward leap in 

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Same-Window vs Time-Aware (The Primary Offender): Same-window validation on client-grouped hash folds looked strong (forest mean P@50 0.672 vs rule 0.376). But the model was partly riding calendar-window correlations: under the time-aware walk-forward (train Dec-Jan, test March) the mean fold fell to 0.260 — a 41-point drop, essentially at the whole-frame base rate (0.249). The ranking learned one window's volatility, not durable decay signals.  

Label-Derived Features: I successfully avoided the label trap by strictly excluding trend_pct and trend_direction, ensuring the label was not circularly computed from a feature column.  

Decision-Derived Features: No product flags or pre-existing system scores were used as inputs; they are correctly reserved strictly as baselines to beat, not as features.  

Overlapping Windows: All features (like imp_prev30) were strictly measured prior to the label's timeline to ensure the model couldn't look into the future.

### Attack Checklist (hunting-leakage-and-validating skill)

- [x] Timeline drawn: all features strictly before the label window
- [x] No label-derived or sibling columns in the features (trend_pct excluded)
- [x] No product flags / existing-system scores as features
- [x] Split grouped by the repeating entity (client_hash_id via deterministic hash folds)
- [x] Base rate printed next to every metric (0.249 whole-frame; fold means shown above)
- [x] Top feature importance sanity-checked — days_with_imp_prev30 is plausible, not suspiciously perfect
- [x] Metrics recomputed out-of-fold (hash folds, not in-sample)

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original Bold Claim: "Our Random Forest reaches 0.67 Precision@50 — nearly triple the baseline rule — so we can ship it to pick next month's refresh queue."

Rewritten Honest Claim: "We observed a mean same-window Precision@50 of 0.672 across five client-grouped folds, versus 0.376 for the transparent rule and a 0.249 whole-frame base rate. Under a time-aware walk-forward audit the mean fold fell to 0.260 — essentially random — so we claim no forward predictive value. The shipped product is the transparent rule with human review; the model's ranking is directional, current-period decision-support only."